In [3]:
import os, json, time, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, matthews_corrcoef, confusion_matrix)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

assert tf.config.list_physical_devices("GPU"), "NO GPU — stop and fix"

EPOCHS, BATCH_SIZE, TEST_SIZE = 40, 512, 0.30
LR, MOMENTUM = 0.01, 0.9
RESULTS_PATH = "/kaggle/working/rescued_perclass.json"

hits = glob.glob("/kaggle/input/**/ciciot2023_working_set.parquet", recursive=True)
df = pd.read_parquet(hits[0])
feature_cols = [c for c in df.columns if c != "family"]
X_all = df[feature_cols].to_numpy(dtype=np.float32)
le = LabelEncoder(); y_all = le.fit_transform(df["family"].to_numpy())
CLASS_NAMES = list(le.classes_)
N_FEATURES, N_CLASSES = X_all.shape[1], len(CLASS_NAMES)
print(f"X: {X_all.shape} | {N_CLASSES} classes: {CLASS_NAMES}")

def build_model(n_inputs, n_output, loss_fn):
    nb = int(round(n_inputs / 2.0))
    visible = keras.Input(shape=(n_inputs, 1))
    e = layers.Dense(n_inputs)(visible); e = layers.BatchNormalization()(e); e = layers.LeakyReLU()(e)
    bn = layers.Dense(nb)(e)
    d = layers.Dense(n_inputs)(bn); d = layers.BatchNormalization()(d); d = layers.LeakyReLU()(d)
    lstm = layers.LSTM(nb, activation="tanh", return_sequences=True)(visible)
    lstm = layers.Dense(n_inputs)(lstm)
    c = layers.Concatenate()([d, lstm])
    c = layers.Conv1D(filters=nb, kernel_size=2, activation="relu")(c)
    c = layers.Flatten()(c)
    out = layers.Dense(n_output, activation="softmax")(c)
    m = keras.Model(visible, out)
    m.compile(optimizer=keras.optimizers.SGD(learning_rate=LR, momentum=MOMENTUM),
              loss=loss_fn, metrics=["accuracy"])
    return m

def categorical_focal_loss(class_weights, gamma=2.0):
    w = tf.constant(class_weights, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)
        return tf.reduce_sum(w * tf.pow(1.0 - y_pred, gamma) * ce, axis=-1)
    return loss

def run(strategy, seed, protocol="B"):
    t0 = time.time()
    np.random.seed(seed); tf.random.set_seed(seed)
    idx = np.arange(len(X_all))
    idx_tr, idx_te = train_test_split(idx, test_size=TEST_SIZE,
                                      random_state=seed, stratify=y_all)
    sc = StandardScaler().fit(X_all[idx_tr])
    X_tr, y_tr = sc.transform(X_all[idx_tr]), y_all[idx_tr]
    X_te, y_te = sc.transform(X_all[idx_te]), y_all[idx_te]

    i_fit, i_val = train_test_split(np.arange(len(y_tr)), test_size=0.10,
                                    random_state=seed, stratify=y_tr)
    X_fit, y_fit = X_tr[i_fit], y_tr[i_fit]
    X_val, y_val = X_tr[i_val], y_tr[i_val]

    if strategy == "focal":
        cnt = np.bincount(y_fit, minlength=N_CLASSES).astype(np.float64)
        cnt[cnt == 0] = 1.0
        cw = cnt.sum() / (N_CLASSES * cnt); cw = cw / cw.mean()
        loss_fn = categorical_focal_loss(cw.astype(np.float32))
    else:
        loss_fn = "categorical_crossentropy"

    rs = lambda a: a.reshape(-1, N_FEATURES, 1).astype(np.float32)
    X_fit, X_val, X_te_r = rs(X_fit), rs(X_val), rs(X_te)
    y_fit_oh = keras.utils.to_categorical(y_fit, N_CLASSES)
    y_val_oh = keras.utils.to_categorical(y_val, N_CLASSES)

    ckpt = f"/kaggle/working/_ck_{strategy}_{seed}.weights.h5"
    model = build_model(N_FEATURES, N_CLASSES, loss_fn)
    hist = model.fit(X_fit, y_fit_oh, epochs=EPOCHS, batch_size=BATCH_SIZE,
                     verbose=0, validation_data=(X_val, y_val_oh),
                     callbacks=[keras.callbacks.ModelCheckpoint(
                         ckpt, monitor="val_accuracy", mode="max",
                         save_best_only=True, save_weights_only=True)])

    def ev(m):
        p = m.predict(X_te_r, batch_size=2048, verbose=0).argmax(1)
        cm = confusion_matrix(y_te, p)
        return {"accuracy": float(accuracy_score(y_te, p)),
                "balanced_accuracy": float(balanced_accuracy_score(y_te, p)),
                "macro_f1": float(f1_score(y_te, p, average="macro", zero_division=0)),
                "mcc": float(matthews_corrcoef(y_te, p)),
                "confusion_matrix": cm.tolist()}

    fin = ev(model)
    model.load_weights(ckpt)
    res_ = ev(model)
    os.remove(ckpt)

    out = {"strategy": strategy, "protocol": protocol, "seed": seed,
           **{k: fin[k] for k in ["accuracy","balanced_accuracy","macro_f1","mcc"]},
           "confusion_matrix": fin["confusion_matrix"],
           "rescued_accuracy": res_["accuracy"],
           "rescued_balanced_accuracy": res_["balanced_accuracy"],
           "rescued_macro_f1": res_["macro_f1"],
           "rescued_mcc": res_["mcc"],
           "rescued_confusion_matrix": res_["confusion_matrix"],
           "best_val_epoch": int(np.argmax(hist.history["val_accuracy"])) + 1,
           "final_train_loss": float(hist.history["loss"][-1]),
           "wall_sec": round(time.time() - t0, 1)}
    keras.backend.clear_session()
    return out

print("Ready.")

X: (547944, 44) | 8 classes: ['Benign', 'BruteForce', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web']
Ready.


In [4]:
import numpy as np, json, os
from sklearn.metrics import average_precision_score, precision_recall_curve

OUT = "/kaggle/working/threshold_analysis.json"
N_SEEDS = 10
results = json.load(open(OUT)) if os.path.exists(OUT) else []
done = {(r["strategy"], r["seed"]) for r in results}

def analyse(y_prob, y_true):
    out = {}
    for ci, cname in enumerate(CLASS_NAMES):
        b = (y_true == ci).astype(int)
        if b.sum() == 0: continue
        ap = average_precision_score(b, y_prob[:, ci])
        p, r, _ = precision_recall_curve(b, y_prob[:, ci])
        f1 = 2*p*r/np.maximum(p+r, 1e-12)
        pred = y_prob.argmax(1)
        tp = ((pred == ci) & (y_true == ci)).sum()
        argmax_f1 = 2*tp / max((pred == ci).sum() + (y_true == ci).sum(), 1)
        out[cname] = {"pr_auc": float(ap), "best_f1": float(f1.max()),
                      "argmax_f1": float(argmax_f1)}
    return out

grid = [(s, sd) for s in ["none", "focal"] for sd in range(N_SEEDS)]
for k, (strat, seed) in enumerate(grid, 1):
    if (strat, seed) in done:
        print(f"[{k}/{len(grid)}] skip"); continue
    print(f"[{k}/{len(grid)}] {strat} seed={seed} ...", flush=True)

    # replicate run() but keep probabilities
    np.random.seed(seed); tf.random.set_seed(seed)
    idx = np.arange(len(X_all))
    itr, ite = train_test_split(idx, test_size=TEST_SIZE, random_state=seed, stratify=y_all)
    sc = StandardScaler().fit(X_all[itr])
    Xtr, ytr = sc.transform(X_all[itr]), y_all[itr]
    Xte, yte = sc.transform(X_all[ite]), y_all[ite]
    ifit, ival = train_test_split(np.arange(len(ytr)), test_size=0.10,
                                  random_state=seed, stratify=ytr)
    if strat == "focal":
        c = np.bincount(ytr[ifit], minlength=N_CLASSES).astype(float); c[c==0]=1
        cw = c.sum()/(N_CLASSES*c); cw = cw/cw.mean()
        lf = categorical_focal_loss(cw.astype(np.float32))
    else:
        lf = "categorical_crossentropy"
    rs = lambda a: a.reshape(-1, N_FEATURES, 1).astype(np.float32)
    m = build_model(N_FEATURES, N_CLASSES, lf)
    m.fit(rs(Xtr[ifit]), keras.utils.to_categorical(ytr[ifit], N_CLASSES),
          epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
          validation_data=(rs(Xtr[ival]),
                           keras.utils.to_categorical(ytr[ival], N_CLASSES)))
    yp = m.predict(rs(Xte), batch_size=2048, verbose=0)

    r = {"strategy": strat, "seed": seed, "per_class": analyse(yp, yte)}
    results.append(r); json.dump(results, open(OUT, "w"))
    for c in ["BruteForce", "Web"]:
        d = r["per_class"][c]
        print(f"    {c:11s} PR-AUC={d['pr_auc']:.4f}  argmaxF1={d['argmax_f1']:.4f}  bestF1={d['best_f1']:.4f}")
    keras.backend.clear_session()

print("\nDone ->", OUT)

[1/20] skip
[2/20] skip
[3/20] skip
[4/20] skip
[5/20] skip
[6/20] none seed=5 ...


I0000 00:00:1787949975.209504      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787949975.212693      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


    BruteForce  PR-AUC=0.2098  argmaxF1=0.2392  bestF1=0.2547
    Web         PR-AUC=0.1409  argmaxF1=0.0581  bestF1=0.1964
[7/20] none seed=6 ...
    BruteForce  PR-AUC=0.2420  argmaxF1=0.2611  bestF1=0.2772
    Web         PR-AUC=0.1642  argmaxF1=0.0474  bestF1=0.2053
[8/20] none seed=7 ...
    BruteForce  PR-AUC=0.2586  argmaxF1=0.2800  bestF1=0.2932
    Web         PR-AUC=0.1751  argmaxF1=0.1123  bestF1=0.2064
[9/20] none seed=8 ...
    BruteForce  PR-AUC=0.2263  argmaxF1=0.2793  bestF1=0.2793
    Web         PR-AUC=0.1284  argmaxF1=0.0607  bestF1=0.1705
[10/20] none seed=9 ...
    BruteForce  PR-AUC=0.2496  argmaxF1=0.2708  bestF1=0.2904
    Web         PR-AUC=0.1747  argmaxF1=0.0413  bestF1=0.2163
[11/20] skip
[12/20] skip
[13/20] skip
[14/20] skip
[15/20] skip
[16/20] focal seed=5 ...
    BruteForce  PR-AUC=0.0851  argmaxF1=0.0640  bestF1=0.2036
    Web         PR-AUC=0.1508  argmaxF1=0.1785  bestF1=0.2045
[17/20] focal seed=6 ...
    BruteForce  PR-AUC=0.2399  argmaxF1=0.0722  